# Minimal dataset onboarding example

In [ ]:
import scanpy as sc
import os

from preprocessing_utils import filter_cells_by_pert_effect

## 1. Download and load dataset

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE264667

HepG2 cell line

`wget https://ftp.ncbi.nlm.nih.gov/geo/series/GSE264nnn/GSE264667/suppl/GSE264667%5Fhepg2%5Fraw%5Fsinglecell%5F01.h5ad`

In [ ]:
base_dir = "."
path_to_dataset = os.path.join(base_dir, "GSE264667_hepg2_raw_singlecell_01.h5ad")

adata = sc.read_h5ad(path_to_dataset)

## 2. Format .obs

In [ ]:
## rename columns
# required column: gene_name (gene subject to perturbation)
# required column: batch_num (relevant batch in experiment)
adata.obs.rename(columns={"gene": "condition", "gem_group" : "batch"}, inplace=True)
assert "condition" in adata.obs.columns, "condition column is required (gene symbol of target gene)"
assert "batch" in adata.obs.columns, "batch column is required"

## create boolean control column
# non-targeting is usually the identifier for controls
control_identifier = "non-targeting"
adata.obs["control"] = (adata.obs["condition"] == control_identifier).astype(int)

# append base state to condition name
adata.obs["condition"] = [c + '+ctrl' for c in adata.obs["condition"]]

# rename control
mapper = {k:k for k in adata.obs["condition"].unique()}
mapper[f"{control_identifier}+ctrl"] = "ctrl"
adata.obs["condition"] = adata.obs["condition"].map(mapper)

## check that cell type columns exists
cell_line_name = "hepg2"
adata.obs["cell_type"] = cell_line_name # add if missing
assert "cell_type" in adata.obs.columns, "cell_type column is required"

## 3. Filter perturbations

In [ ]:
_, adata = filter_cells_by_pert_effect(adata)

## 4. Variance stabilizing transform

In [ ]:
sc.pp.normalize_total(adata, target_sum=4000)
sc.pp.log1p(adata)

## 5. Highly variable gene selection

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=5000, subset=True)

Dataset can now be used for training, remaining formatting will be performed by TxPert.

In [ ]:
save_path = "./nadig_hepg2_preprocessed.h5ad"

adata.write(save_path)